In [1]:
import scipy.io
import io

from data_loader import DataLoader

import numpy as np
from utils import *
from matplotlib import pyplot as plt
import numpy.random as rng
import pandas as pd

import os
import math

Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.


In [2]:
def spk_matrix_init(session, start_time, end_time, neuron_ids, session_neurons):
    """
    Initializes a spike matrix for a given time window and set of neurons.

    Parameters:
    session : object
        Session data containing neuronal spike information.
     start_time : int
        Start time of the window for spike extraction.
    end_time : int
        End time of the window for spike extraction.
    neuron_ids : list of int
        Identifiers of the neurons to be included in the matrix.
    session_neurons : dict
        Dictionary mapping neuron IDs to their spike times.

    Returns:
    spike_matrix : ndarray, shape (n_neurons, length)
        Binary matrix indicating spike occurrences within the specified time window.
    """
    length = end_time - start_time
    spike_matrix = np.zeros((n_neurons, length), dtype=int)
    
    for i, neuron_id in enumerate(neuron_ids):
        spike_times = np.floor(session_neurons[neuron_id]).astype(int)
        spike_times_windowed = spike_times[(spike_times >= start_time) & (spike_times < end_time)]- start_time #<- comment out to not make it start at 0
        spike_matrix[i, spike_times_windowed] = 1 #set corresponding columns to 1 where spikes occurred
    return spike_matrix   

In [5]:
def init_spk_matrix_train_safe_runs(train_runs, session, n_neurons, neuron_ids, session_neurons):
    """
    Initializes spike matrices for multiple training runs with varying durations.

    Parameters:
    train_runs : ndarray
        Array containing start, middle, and end times for each training run.
    session : object
        Session data containing neuronal spike information.
    n_neurons : int
        Number of neurons to be included in the spike matrices.
    neuron_ids : list of int
        Identifiers of the neurons to be processed.
    session_neurons : dict
        Dictionary mapping neuron IDs to their spike times.

    Returns:
    spk_matrices : list of ndarray
        List of binary spike matrices, each corresponding to a training run with different sizes.
    """
    spk_matrices = []  #List to store spike matrices of varying sizes

    for i in range(train_runs.shape[0]):
        #Start and end time for each run
        start_time = math.ceil(train_runs[i, 0])  # Middle time
        end_time = math.ceil(train_runs[i, 2])   # End time
        temp_spk_matrix = spk_matrix_init(session, start_time, end_time, neuron_ids, session_neurons)

        spk_matrices.append(temp_spk_matrix)

    return spk_matrices

#Remember this is a list of matrices and not a matrix! (they have a different size)

In [4]:
def plot_spk_matrix(spk_matrix):
    for i in range(spk_matrix.shape[0]):
        plt.plot(np.where(spk_matrix[i,:] == 1)[0], i+np.ones((np.sum(spk_matrix[i,:]),1)), 'k|', markersize=10)
        plt.xlabel('Time in ms')
        plt.ylabel('Neuron ids')

In [6]:
def spk_matrix_window_init(session, center_time, window_size):
    """
    Initializes a spike matrix for a specified time window centered around a given time.
    Parameters:
    session : object
        Session data containing neuronal spike information.
    center_time : int
        The central time point around which the window is defined.
    window_size : int
        The total size of the time window.

    Returns:
    spike_matrix : ndarray, shape (n_neurons, window_size)
        Binary matrix indicating spike occurrences within the specified time window.
    """
    half_window = window_size // 2
    window_start = int(center_time - half_window)
    window_end = int(center_time + half_window)

    spike_matrix = np.zeros((n_neurons, window_end - window_start), dtype=int)

    for i, neuron_id in enumerate(neuron_ids):
        spike_times = np.floor(session_neurons[neuron_id]).astype(int)
        spike_times_windowed = spike_times[(spike_times >= window_start) & (spike_times < window_end)] - window_start
        spike_matrix[i, spike_times_windowed] = 1  
    return spike_matrix

In [7]:
def analyze_neuron_spikes_window(spk_matrix_train, neuron_idx, window_size, session):
    """
    Extract a spike matrix for all neurons around each spike time of a specific neuron
    with a window of ±window_size (in ms).

    Parameters:
    spk_matrix_train : list of ndarray
        List of spike matrices for each trial, with shape (n_neurons, time).
    neuron_idx : int
        Index of the neuron whose spike-triggered windows will be analyzed.
    window_size : int
        Size of the time window (in milliseconds) to extract around each spike.
    session : object
        Session data containing neuronal spike information.

    Returns:
    spk_matrix_window_3d : ndarray, shape (n_spikes, n_neurons, window_size)
        3D array containing extracted spike matrices for each detected spike, 
        padded if necessary.
    """

    half_window = window_size // 2
    windowed_spike_matrices = [] 

    num_trials = len(spk_matrix_train)

    for trial_idx in range(num_trials):
        # Get the spike matrix for the current trial
        trial_spk_matrix = spk_matrix_train[trial_idx]  # Shape: (n_neurons, time)

        # Extract the spike data for the target neuron in this trial
        neuron_spikes = trial_spk_matrix[neuron_idx, :]  # Shape: (time,)

        # Identify spike times (non-zero entries)
        spike_times = np.where(neuron_spikes > 0)[0]  # Spike indices (in ms)
        for spike_time in spike_times:
            # Define the start and end of the window
            window_start = max(0, spike_time - half_window)  # Handle edge case at the start
            window_end = min(trial_spk_matrix.shape[1], spike_time + half_window)  # Handle edge case at the end

            # Extract the spike matrix slice for all neurons in this window
            spk_matrix_window = trial_spk_matrix[:, window_start:window_end]  # Shape: (n_neurons, window_size)

            # Pad if necessary (e.g., if near trial boundaries)
            if spk_matrix_window.shape[1] < window_size:
                padded_matrix = np.zeros((trial_spk_matrix.shape[0], window_size))
                offset = (window_size - spk_matrix_window.shape[1]) // 2
                padded_matrix[:, offset:offset + spk_matrix_window.shape[1]] = spk_matrix_window
                spk_matrix_window = padded_matrix

            # Store the spike matrix for this spike time
            windowed_spike_matrices.append(spk_matrix_window)
        if len(windowed_spike_matrices) > 0:
            spk_matrix_window_3d = np.stack(windowed_spike_matrices, axis=0)  # Combine into 3D array
        else:
            # Return an empty array with shape (0, n_neurons, window_size) if no spikes
            n_neurons = spk_matrix_train[0].shape[0]  # Number of neurons
            spk_matrix_window_3d = np.zeros((0, n_neurons, window_size))

    return spk_matrix_window_3d

**Build the ranks from the spike matrix**

In [7]:
def spk_matrix_to_ranks(spk_matrix, trigger_neuron_idx): #output is the neuron idx in the place related to the trigger neuron
    n_trials, n_neurons = spk_matrix.shape[0], spk_matrix.shape[1]
    ranks = np.full((n_trials, n_neurons*2+1), np.nan)  # Shape: [trials x neurons*2+1]
    for trial in range(n_trials):
        avg_spk_time = []
        for neuron_idx in range(n_neurons):
            spike_times = np.where(spk_matrix[trial, neuron_idx] == 1)[0]  
            if len(spike_times) > 0:  
                avg_spk_time.append(np.mean(spike_times))  
            else:
                avg_spk_time.append(np.nan)  

        avg_spk_time = np.array(avg_spk_time) 
        neuron_sorted = np.argsort(avg_spk_time)
        
        valid_neurons = [neuron_idx for neuron_idx in neuron_sorted if not np.isnan(avg_spk_time[neuron_idx])]

        if trigger_neuron_idx in valid_neurons:
            base_rank = valid_neurons.index(trigger_neuron_idx) # base rank gets index of trigger neuron in valid neurons
            ranks[trial, n_neurons] = trigger_neuron_idx  # Trigger neuron rank is 0
        else:
            base_rank = None
              

        for relative_rank, neuron_idx in enumerate(valid_neurons): #relative rank is index in valid neurons 
            if base_rank is not None:  
                idx = relative_rank - base_rank + n_neurons
                ranks[trial, idx] = neuron_idx
            else:  
                ranks[trial, :] = np.nan # hier dus nog goed naar kijken!!!
    return ranks

**Compute OCC for the ranks**

In [8]:
def firing_ranks2occ_matrix(firing_ranks):
    """
    Function adapted from Tom Has.
    Constructs an occurrence matrix from firing ranks where occ[x, y] represents how often neuron x had rank y.
    Input:  - firing_ranks: Array containing for each rank in each trial which neuron had this rank in this trial
    Output: - occ:          Occurrence matrix
    """
    n_neurons = firing_ranks.shape[1]
    occ = np.zeros((n_neurons, n_neurons), dtype=int)
    # Loop over trials
    for trial in firing_ranks:
        # Loop over ranks:
        for rank, neuron in enumerate(trial):
            if np.isnan(neuron):
                continue
            # For not nan rank, neuron pairs, increment the respective square in the occurrence matrix|
            occ[int(neuron), rank] += 1
    return occ

In [10]:
def MI(M):
    """
    Function adopted from Tom Has.
    Calculates the mutual information between the 2 axes of a matrix.
    Input:  - M:  2d matrix
    Output: - MI: Mutual information
    """
    sizex = M.shape[0]
    sizey = M.shape[1]
    total = np.sum(M)
    p_Y = np.sum(M, axis=0) / total
    p_X = np.sum(M, axis=1) / total
    # MI = sum over x and y: p(x,y) * log(p(x,y) / (p(x) * p(y)))
    #    = sum over x and y: M(x,y)/total * log((M(x,y) * total) / (p_X(x) * p_Y(y)))
    MI = sum([sum([(
        0 if M[x, y] == 0 else
        M[x, y] / total * np.log((M[x, y] / total) / (p_X[x] * p_Y[y]))
    ) for x in range(sizex)]) for y in range(sizey)])
    return MI

**Plot the OCC**

In [8]:
def sort_occ_matrix(occ_matrix, mean_rank=False):   
    """"
    Function adopted from Pepijn van den Berg (2024).
    Modifies an occ matrix to only active neurons and 'used' rank positions. 
    Also, it sorts the matrix, either based on mean rank, or on mode rank.
    """
    # Find active neurons
    active_neurons = np.where(np.any(occ_matrix>0, axis=1))[0]

    # Find non-zero rank range
    active_ranks = np.where(np.any(occ_matrix!=0, axis=0))[0]
    if len(active_ranks)<=1:
        rank_limit_1 = 1
        rank_limit_2 = 2
    else:
        rank_limit_1 = active_ranks[1]
        rank_limit_2 = active_ranks[-1]

    # Pick 'active' rows of occ_matrix, for relevant rank range
    occ_matrix = occ_matrix[active_neurons, rank_limit_1:rank_limit_2+1]
    
    # Compute max rank
    max_ranks = np.argmax(occ_matrix, axis=1)
    
    # Compute mean rank
    mean_ranks = np.zeros(len(active_neurons))
    for ineuron, neuron in enumerate(active_neurons):
        mean_ranks[ineuron] = np.mean(occ_matrix[ineuron,:]*np.arange(rank_limit_1, rank_limit_2+1))
    
    
    # Return sorted occ matrix
    if mean_rank:
        return [occ_matrix[np.argsort(mean_ranks),:], np.argsort(mean_ranks), active_neurons]
    else:
        return [occ_matrix[np.argsort(max_ranks),:], np.argsort(max_ranks), active_neurons]

**Code for obtaining the surrogates**

In [24]:
def shuffle_ranks(ranks, trigger_neuron_idx):
    """
    Randomly shuffles the ranks of neurons while preserving the trigger neuron's position.
    """
    shuffled_ranks = ranks.copy()
    n_trials, n_neurons = ranks.shape[0], (ranks.shape[1]-1)//2

    for trial in range(n_trials):
        active_neuron_idxs = [idx for idx, val in enumerate(ranks[trial]) if not np.isnan(val) and idx != n_neurons]
        active_neurons = [ranks[trial, idx] for idx in active_neuron_idxs]
    
        rng.shuffle(active_neurons)
    
        for idx, neuron in zip(active_neuron_idxs, active_neurons):
            shuffled_ranks[trial, idx] = neuron

        shuffled_ranks[trial, trigger_neuron_idx] = ranks[trial, trigger_neuron_idx]
    
    return shuffled_ranks

In [13]:
def surrogate_MI(trial_matrix, n_iterations):
    """
    Generates surrogate MI distributions for a given trigger neuron 
    by shuffling spike rank data and computing occurrence matrices.

    Input:  
    - trial_matrix: ndarray
        Spike matrix representing neuronal activity across trials, with shape (n_neurons, time).
    - n_iterations: int
        Number of iterations to generate surrogate data.
    - trigger_neuron: int
        Index of the neuron used as the trigger for rank calculations.
    """
    surrogate_mi = np.empty(n_iterations, dtype=float)
    for i in range(n_iterations):
        shuffled = shuffled_ranks(trial_matrix)
        occ = firing_ranks2occ_matrix(shuffled)
        surrogate_mi[i] = MI(occ)
    return surrogate_mi

In [14]:
def surrogate_MI_trigger_neuron(trial_matrix, n_iterations, trigger_neuron):
    """
    Generates surrogate MI distributions for a given trigger neuron 
    by shuffling spike rank data and computing occurrence matrices.

    Input:  
    - trial_matrix: ndarray
        Spike matrix representing neuronal activity across trials, with shape (n_neurons, time).
    - n_iterations: int
        Number of iterations to generate surrogate data.
    - trigger_neuron: int
        Index of the neuron used as the trigger for rank calculations.

    Output: 
    - surrogate_mi: ndarray
        Array containing mutual information values computed from surrogate data.
    - significance_boundary: float
        The 95th percentile threshold of surrogate mutual information values.
    """
    surrogate_mi = np.empty(n_iterations, dtype=float)  
    ranks = spk_matrix_to_ranks(trial_matrix, trigger_neuron)
    for i in range(n_iterations):
        shuffled = shuffle_ranks(ranks, trigger_neuron)
        occ = firing_ranks2occ_matrix(shuffled)
        surrogate_mi[i] = MI(occ)
    significance_boundary = np.percentile(surrogate_mi, 95)
    return surrogate_mi, significance_boundary

In [15]:
def plot_surrogate(MI_surrogate, threshold):
    plt.hist(MI_surrogate, bins=20, color='gray', alpha=0.7, label='Surrogate MI')
    plt.axvline(threshold, color='blue', linestyle='dashed', linewidth=2, label=f'95th Percentile: {threshold:.2f}')
    plt.xlabel('Mutual Information')
    plt.ylabel('Frequency')
    plt.title('Distribution of Surrogate MI Values')
    plt.legend()
    plt.show()

In [16]:
def MI_trigger_neuron(spk_matrix, trigger_neuron_idx, n_iterations):
        """
    Computes the MI for a given trigger neuron and generates 
    surrogate distributions to assess significance.
    Input:  
    - spk_matrix: ndarray
        Binary spike matrix with shape (time, n_neurons), where each row represents 
        a time point and columns correspond to neuron activities.
    - trigger_neuron_idx: int
        Index of the neuron used as the trigger for rank calculations.
    - n_iterations: int
        Number of surrogate iterations for significance testing.

    Output: 
    - MI_actual: float
        The mutual information value computed from the actual spike data.
    - occ_actual: ndarray
        The occurrence matrix computed from the real spike data.
    - surrogate_mi: ndarray
        Array of mutual information values computed from surrogate data.
    - significance_boundary: float
        The 95th percentile threshold of surrogate mutual information values.
    """
    ranks = spk_matrix_to_ranks(spk_matrix, trigger_neuron_idx)
    occ_actual = aligned_firing_ranks2occ_matrix(ranks)
    MI_actual = MI(occ_actual)
    
    surrogate_mi = np.empty(n_iterations, dtype=float)
    for i in range(n_iterations):
        shuffled_ranks = shuffle_ranks(ranks, trigger_neuron_idx)
        occ_surrogate = aligned_firing_ranks2occ_matrix(shuffled_ranks)
        surrogate_mi[i] = MI(occ_surrogate)
    
    significance_boundary = np.percentile(surrogate_mi, 95)
    
    return MI_actual, occ_actual, surrogate_mi, significance_boundary

In [21]:
def surrogate_MI_neurons(spk_matrix, n_iterations):
    """
    Computes MI and surrogate distributions for all neurons.

    Parameters:
    spk_matrix : ndarray
        Binary spike matrix with shape (time, n_neurons), where each row represents 
        a time point and columns correspond to neuron activities.
    n_iterations : int
        Number of iterations to generate surrogate data.
    """
    n_neurons = spk_matrix.shape[1]
    MI = np.empty(n_neurons)
    MI_surrogate = np.empty((n_neurons, n_iterations))
    sig_threshold = np.empty(n_neurons)
    occ_per_trigger = np.zeros((n_neurons, n_neurons+1, n_neurons*2))
    for n in range(n_neurons): 
        MI[n], occ_per_trigger[n], MI_surrogate[n], sig_threshold[n] = MI_trigger_neuron(spk_matrix, n, n_iterations)
        
    return MI, occ_per_trigger, MI_surrogate, sig_threshold

In [18]:
def plot_mi_values(mi_matrix, neuron_ids):
    plt.figure(figsize=(10, 6))
    plt.stem(mi_matrix[:, 0], mi_matrix[:, 1], basefmt=" ")
    plt.xlabel("Neuron IDs")
    plt.ylabel("Mutual Information (MI)")
    plt.title("Mutual Information for Neurons")
    plt.grid(True)
    plt.show()

In [19]:
def get_significant_neurons(mi_values, boundary):
    """
    Identifies significant neurons based on MI values and boundary thresholds.
    Parameters:
    mi_values : ndarray
        Array of mutual information values for each neuron.
    boundary : ndarray
        Array of threshold values for determining significance.
    Returns:
    significant_neurons : ndarray
        Array containing mutual information values for significant neurons, 
        with NaN for non-significant neurons.
    """
    significant_neurons = np.full((len(mi_values)), np.nan)
    for idx in range(len(mi_values)):
        mi_neuron = mi_values[idx]
        bound = boundary[idx]
        if mi_neuron >= bound and mi_neuron != 0:
            significant_neurons[idx] = mi_neuron
    return significant_neurons

**Compute MI, OCC, surrogates with every neuron as trigger neuron**

In [20]:
def compute_mi_occ_surrogates_per_neuron(spk_matrix_train, window_size, n_iterations):
    """
    Computes the occurrence matrix, mutual information, and surrogate values for each neuron.

    Parameters:
    spk_matrix_train : list of ndarray
        List of spike matrices for each trial, with shape (n_neurons, time).
    window_size : int
        Size of the time window (in milliseconds) to extract around each spike.
    n_iterations : int
        Number of surrogate iterations for mutual information computation.

    Returns:
    occ_matrix : ndarray
        3D array containing occurrence matrices for each neuron.
    real_mi : ndarray
        Array of real mutual information values for each neuron.
    surrogates_per_neuron : ndarray
        Array of surrogate mutual information values for each neuron across iterations.
    boundary_per_neuron : ndarray
        Array of boundary values derived from surrogate distributions for each neuron.
    """
    #initialize the variable to save in
    n_neurons = len(spk_matrix_train[0])
    occ_matrix = np.zeros((n_neurons, n_neurons*2+1, n_neurons*2+1))
    real_mi = np.zeros((n_neurons))
    surrogates_per_neuron = np.empty((n_neurons, n_iterations))
    boundary_per_neuron = np.zeros((n_neurons,))
    
    for neuron_idx in range(n_neurons):
        print(f"Analyzing neuron {neuron_idx} of {n_neurons}")
        #create a window around every spike of the specific trigger neuron
        spk_matrix_for_trigger_neuron = analyze_neuron_spikes_window(
            spk_matrix_train, neuron_idx, window_size, session)  # Shape: (n_spikes, n_neurons, window_size)

         # If there are no spikes for this neuron, skip OCC and MI computation
        if spk_matrix_for_trigger_neuron.shape[0] == 0:
            occ_matrix[neuron_idx] = 0
            real_mi[neuron_idx] = 0
            continue

        ranks = spk_matrix_to_ranks(spk_matrix_for_trigger_neuron, neuron_idx)
        occ = firing_ranks2occ_matrix(ranks)
        mi = MI(occ)
        surrogates, boundary = surrogate_MI_trigger_neuron(spk_matrix_for_trigger_neuron, n_iterations, neuron_idx)
        print(f"Computed ranks, occ, mi and surrogates for  {neuron_idx} of {n_neurons}")
        #store values
        occ_matrix[neuron_idx] = occ
        real_mi[neuron_idx] = mi
        surrogates_per_neuron[neuron_idx] = surrogates
        boundary_per_neuron[neuron_idx] = boundary

    return occ_matrix, real_mi, surrogates_per_neuron, boundary_per_neuron

In [20]:
def plot_surrogate(surrogates, boundary, mi_values, ax, neuron_ids):
    """
    Plots the surrogate MI values with boundary and real MI.

    Parameters:
    surrogates : ndarray
        Array of surrogate mutual information values.
    boundary : float
        Threshold value representing the 95th percentile of surrogate distribution.
    mi_values : float
        Real mutual information value to compare against surrogates.
    ax : matplotlib.axes.Axes
        Matplotlib axis object to plot on.
    neuron_ids : int
        ID of the neuron being analyzed.
    """
    ax.hist(surrogates, bins=20, color='gray', alpha=0.7, label='Surrogate MI')
    ax.axvline(boundary, color='grey', linestyle='dashed', linewidth=2, label=f'95th Percentile: {boundary:.2f}')
    ax.axvline(mi_values, color='red', linestyle='dashed', linewidth=2, label=f'Real MI: {mi_values:.2f}')
    ax.set_xlabel('Mutual Information')
    ax.set_ylabel('Frequency')
    ax.set_title(f'Surrogate MI Values oaf neuron {neuron_ids}')
    ax.legend()

def plot_z_surrogate(surrogates, boundary, mi_values, ax, neuron_ids):
    """
    Plots the z-scored surrogate mutual information values with real MI.

    Parameters:
    surrogates : ndarray
        Array of surrogate mutual information values.
    boundary : float
        Threshold value for comparison (not used in plotting).
    mi_values : float
        Real mutual information value for comparison.
    neuron_ids : int
        ID of the neuron being analyzed.
    """
    try:
        ax.hist(zscore_by_surr(surrogates,surrogates), bins=20, color='gray', alpha=0.7, label='Surrogate MI')
        # ax.axvline(boundary, color='grey', linestyle='dashed', linewidth=2, label=f'95th Percentile: {boundary:.2f}')
        ax.axvline(zscore_by_surr(mi_values,surrogates), color='red', linestyle='dashed', linewidth=2, label=f'Real MI: {mi_values:.2f}')
        ax.set_xlabel('Mutual Information')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Surrogate MI Values of neuron {neuron_ids}')
        ax.legend()
    except:
        print('Empty?')
def plot_all_zsurrogate(real_mi, surrogates, boundary, neuron_ids):
    """
    Plots z-scored surrogate mutual information values for multiple neurons.

    Parameters:
    real_mi : list of float
        List of real mutual information values.
    surrogates : list of ndarray
        List of surrogate mutual information values for each neuron.
    boundary : list of float
        List of boundary values for each neuron.
    neuron_ids : list of int
        List of neuron IDs being analyzed.
    Returns: nothing

    """
    num_plots = len(neuron_ids)
    cols = 5
    rows = (num_plots + cols - 1) // cols  # Ceiling division
    
    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=(30, 20))
    axes = axes.flatten()  # Flatten to make indexing easier
    
    # Loop through and plot
    for n in range(num_plots):
        plot_z_surrogate(surrogates[n], boundary[n], mi_real[n], axes[n], neuron_ids[n])
    
    # Hide unused subplots
    for ax in axes[num_plots:]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()


def plot_all_surrogate(real_mi, surrogates, boundary, neuron_ids):
    """
    Plots surrogate mutual information values for multiple neurons.

    Parameters:
    real_mi : list of float
        List of real mutual information values.
    surrogates : list of ndarray
        List of surrogate mutual information values for each neuron.
    boundary : list of float
        List of boundary values for each neuron.
    neuron_ids : list of int
        List of neuron IDs being analyzed.
    Returns: nothing
    """
    nm_plots = len(neuron_ids)
    cols = 5
    rows = (num_plots + cols - 1) // cols  # Ceiling division
    
    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=(30, 20))
    axes = axes.flatten()  # Flatten to make indexing easier
    
    # Loop through and plot
    for n in range(num_plots):
        plot_surrogate(surrogates[n], boundary[n], mi_real[n], axes[n], neuron_ids[n])
    
    # Hide unused subplots
    for ax in axes[num_plots:]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [22]:
def zscore_by_surr(real_MI, surr_MI):
    m = np.mean(surr_MI)
    s = np.std(surr_MI)
    real_MI = (real_MI-m)/s
    return real_MI

In [23]:
def plot_occ_big(occ):
    def plot_occs(occ_matrix, ax, neuron_idx):
        ax.imshow(sort_occ_matrix(occ_matrix)[0], aspect='auto', origin='lower')
        ax.set_xlabel('Occurence')
        ax.set_ylabel('Neuron IDX')
        ax.set_title(f'OCC matrix with trigger_neuron {neuron_idx}')
        ax.legend()
    
    num_plots = len(occ)
    cols = 5
    rows = (num_plots + cols - 1) // cols  # Ceiling division
    
    # Create subplots
    fig, axes = plt.subplots(rows, cols, figsize=(20, 20))
    axes = axes.flatten()  # Flatten to make indexing easier
    
    # Loop through and plot
    for i in range(len(occ)):
        plot_occs(occ[i], axes[i], [i])
    
    # Hide unused subplots
    for ax in axes[num_plots:]:
        ax.axis('off')
    
    plt.tight_layout()
    plt.show()

In [11]:
def get_sequence_from_occ(original_mi, original_boundary, occ_list):
    """
    Determines the sequence of neuron removals based on MI
    and significance thresholding.

    Parameters:
    original_mi : ndarray
        Array containing the original mutual information values for neurons
    original_boundary : float
        Threshold used to determine the significance of neurons.
    occ_list : list of ndarray
        List of occurrence arrays where each entry corresponds to a neuron’s occurrence.
    Returns:
    removed_neurons : list of int
        List of neuron indices removed in the order of their significance.
    """
    significant_neurons = get_significant_neurons(original_mi, original_boundary)
    removed_neurons = []
    list_of_remaining_occ = occ_list.copy()
    while significant_neurons.size > 0:
        if np.all(np.isnan(significant_neurons) | (significant_neurons == 0)):
            print("No more significant neurons.")
            break

        neuron_to_delete_idx = np.nanargmax(significant_neurons)
        
        removed_neurons.append(neuron_to_delete_idx)

        new_MI = np.empty(len(original_mi))
        for i, occ_value in enumerate(list_of_remaining_occ):
            list_of_remaining_occ[i][ neuron_to_delete_idx] = 0
            list_of_remaining_occ[neuron_to_delete_idx] = 0 
            new_MI[i] = MI(occ_value)

        significant_neurons = get_significant_neurons(new_MI, original_boundary)
       
    return removed_neurons    
            

**Sequence based on rank distribution - not used for analysis**

In [9]:

def get_rank_distribution(sig_neur, occs):
    """
    Computes the rank distribution of significant neurons based on their occurrences 
    in the provided occurrence matrix.

    Parameters:
    sig_neur : ndarray
        Array indicating significant neurons, with NaN values for non-significant ones
    occs : ndarray
        2D array representing the occurrences of neurons, where each row corresponds 
        to a neuron and columns indicate ranked neuron occurrences.

    Returns:
    rank_matrix : ndarray, shape (n_significant_neurons, n_significant_neurons)
        Matrix containing the ranks of significant neurons in the occurrence matrix, 
        with NaN values for missing occurrences.
    """
    indices_significant_neurons = np.where(~np.isnan(significant_neurons))[0] 
    n_neurons = len(indices_significant_neurons)
    rank_matrix = np.full((n_neurons, n_neurons), np.nan)  # Initialize rank matrix with NaNs
    
    for i, viewed_neuron in enumerate(indices_significant_neurons):
        other_neurons=[]
        for j, other_neuron in enumerate(indices_significant_neurons):
            if i != j:  # Only check other neurons
                other_neurons.append(other_neuron)
                # Get the occurrence row of the other_neuron
                occ_other_neuron = occs[other_neuron]
                
                # Find the rank (column index) of the viewed_neuron in the other_neuron's occ
                ranks = np.where(occ_other_neuron == viewed_neuron)[0]  # Get column indices where it appears
                # print(f"i{i} and j{j} give ranks{ranks}")
                if len(ranks) > 0:
                    rank_matrix[i, j] = ranks[0]  # Take the first occurrence (or handle duplicates)
        # print(f"For viewd_neuron{viewed_neuron}, these other neurons are investigated {other_neurons}")
    return rank_matrix

In [10]:
def compute_significant_neuron_ranks(occ, significant_neurons):
    """
    Computes a 2D matrix where each row corresponds to a significant neuron,
    and each value in that row is the rank (with the highest occurrence) of the neuron
    in the occurrence (occ) data of the other significant neurons.

    Parameters:
    - occ: 2D NumPy array where rows correspond to neuron indices and columns to ranks.
    - significant_neurons: List of indices of significant neurons.

    Returns:
    - 2D NumPy array where each row corresponds to a significant neuron and contains
      the ranks with the highest occurrence in the occ of each other significant neuron.
    """

    indices_significant_neurons = np.where(~np.isnan(significant_neurons))[0] 
    # Initialize the result matrix
    num_significant_neurons = len(indices_significant_neurons)
    result_matrix = np.full((num_significant_neurons, num_significant_neurons), np.nan)

    # Calculate the middle column (rank 0)
    middle_col = occ.shape[1] // 2
    print(middle_col)

    for i, viewed_neuron in enumerate(indices_significant_neurons):
        for j, other_neuron in enumerate(indices_significant_neurons):
            if i == j:
                # Skip the self-comparison
                continue

            # Get the occurrence row of the other neuron
            occ_row = occ[other_neuron, :]

            # Find the column with the highest occurrence of the viewed neuron
            viewed_neuron_occurrences = np.where(occ_row == viewed_neuron)[0]

            if len(viewed_neuron_occurrences) > 0:
                # Translate the column index to the rank
                rank_with_highest_occurrence = np.max(viewed_neuron_occurrences) - middle_col

                # Store the rank in the result matrix
                result_matrix[i, j] = rank_with_highest_occurrence

    return result_matrix



In [27]:
def get_significant_ranks(occ, significant_neurons):
    """
    Calculate the highest occurrence rank for each significant neuron
    in the occurrence matrix (occ) of every other significant neuron.

    Parameters:
    - occ (np.ndarray): 2D array where rows are neuron indices and columns are rank occurrences.
    - significant_neurons (list): List of indices for significant neurons.

    Returns:
    - rank_matrix (np.ndarray): 2D array where each row corresponds to a significant neuron, and each column
                                contains the rank with the highest occurrence in the occ of every other neuron.
    """

    indices_significant_neurons = np.where(~np.isnan(significant_neurons))[0] 
    # Number of significant neurons
    num_significant_neurons = len(indices_significant_neurons)
    
    # Determine the middle column index (where rank 0 is located)
    middle_index = occ.shape[1] // 2
    
    # Initialize the rank matrix
    rank_matrix = np.full((num_significant_neurons, num_significant_neurons), np.nan)
    
    # Loop through each significant neuron as the "viewed neuron"
    for i, viewed_neuron in enumerate(indices_significant_neurons):
        # Loop through each other significant neuron to check their occurrences
        for j, other_neuron in enumerate(indices_significant_neurons):
            # Skip if the viewed neuron is the same as the other neuron
            if viewed_neuron == other_neuron:
                continue
            
            # Get the occ row of the other neuron
            occ_row = occ[other_neuron]
            
            # Find the indices where the viewed neuron has occurrences
            viewed_occ_indices = np.where(occ_row == viewed_neuron)[0]
            
            if len(viewed_occ_indices) > 0:
                # Count the occurrences of each rank
                rank_counts = {}
                for idx in viewed_occ_indices:
                    # Calculate the rank based on middle_index
                    rank = idx - middle_index
                    rank_counts[rank] = rank_counts.get(rank, 0) + 1
                
                # Find the rank with the highest occurrence
                highest_occurrence_rank = max(rank_counts, key=rank_counts.get)
                
                # Store the rank in the matrix
                rank_matrix[i, j] = highest_occurrence_rank
    
    return rank_matrix, indices_significant_neurons

In [28]:
def plot_ranks_distribution(mean_ranks, indices):
    
    # Create the plot
    plt.figure(figsize=(10, 4))
    y_position = 0  # All points share the same y-axis value
    
    # Scatter plot
    plt.scatter(mean_ranks, [y_position] * len(mean_ranks), color='blue', zorder=2)
    
    # Annotate each point with its index
    for i, (x, idx) in enumerate(zip(mean_ranks, indices)):
        plt.text(x, y_position + 0.05, f"{idx}", fontsize=6, ha='center', color='black')
    
    # Labeling
    plt.xlabel('Rank with Highest Occurrence', fontsize=12)
    plt.title('Highest Occurrence Ranks of Significant Neurons', fontsize=14)
    
    # Y-axis settings
    plt.yticks([y_position], [''])  # Only one y-tick for alignment
    plt.axhline(y_position, color='gray', linestyle='--', zorder=1)  # Horizontal line for reference
    
    # X-axis grid for clarity
    plt.grid(axis='x', alpha=0.3)
    
    # Show the plot
    plt.tight_layout()
    plt.show()

In [29]:
def mean_ranks_to_seq_with_max_distance (mean_ranks, indices, threshold=2):
    # Include ranks that are within the gap threshold (difference < 2)
    sorted_indices = np.argsort(mean_ranks)  # Sorting order based on ranks

    # Sort the ranks and indices accordingly
    sorted_ranks = mean_ranks[sorted_indices]
    sorted_neuron_indices = indices[sorted_indices]

    # Initialize the filtered ranks list with the first value
    filtered_ranks = [sorted_ranks[0]]
    seq_order_idx = []
    # Loop through sorted ranks and compute differences
    for i in range(len(sorted_ranks) - 1):
        diff = sorted_ranks[i + 1] - sorted_ranks[i]
        seq_order_idx.append(sorted_neuron_indices[i])
        if diff > threshold:
            break  # Stop including further ranks if gap exceeds the threshold
        filtered_ranks.append(sorted_ranks[i + 1])
    
    return np.array(filtered_ranks), np.array(seq_order_idx)
 

In [30]:
def mean_ranks_to_seq(mean_ranks, indices):
    # Include ranks that are within the gap threshold (difference < 2)
    sorted_indices = np.argsort(mean_ranks)  # Sorting order based on ranks

    # Sort the ranks and indices accordingly
    sorted_ranks = mean_ranks[sorted_indices]
    sorted_neuron_indices = indices[sorted_indices]
    return np.array(sorted_ranks), np.array(sorted_neuron_indices)

**Building a template for the sequence**


The idea is to take the signficant neuron with the highest MI, check if that is in the sequence
Then mask all the other neurons (give them -0's that are not in the sequence)

In [31]:
def create_occ_pattern_template(occs, significant_neurons, seq_order):
    """
    Creates an occ template for projection purposes by masking rows not in seq_order.

    Parameters:
    - occs (np.ndarray): The occurrence matrix with shape (n_neurons, n_ranks).
    - significant_neurons (np.ndarray): Array of indices for significant neurons.
    - seq_order (np.ndarray): The order of neurons in the sequence.

    Returns:
    - template (np.ndarray): The modified occ template.
    """
    # Find the neuron with the highest MI value among significant neurons
    neuron_idx = np.nanargmax(significant_neurons)
    
    # Check if the neuron_idx is in the sequence order
    if neuron_idx not in seq_order:
        raise ValueError(f"Neuron {neuron_idx} is not in the sequence order.")
    
    # Copy the occ matrix for the neuron with the highest MI value
    occ_template = occ[neuron_idx].copy()
    
    # Mask rows not in seq_order by setting their values to 0
    for neuron_idx in range(occ_template.shape[0]):
        if neuron_idx not in seq_order:
            occ_template[neuron_idx, :] = 0  # Set all values in the row to 0
    
    return occ_template

In [1]:
def create_occ_pattern_template_probabilities(occs, significant_neurons, seq_order):
    """
    Creates an occurrence probability template for projection purposes by masking rows not in seq_order
    and converting occurrences to probabilities. Rows with only zeros are left unchanged.

    Parameters:
    - occs (np.ndarray): The occurrence matrix with shape (n_neurons, n_ranks).
    - significant_neurons (np.ndarray): Array of indices for significant neurons.
    - seq_order (np.ndarray): The order of neurons in the sequence.

    Returns:
    - template (np.ndarray): The modified occurrence probability template.
    """
    # Find the neuron with the highest MI value among significant neurons
    neuron_idx = np.nanargmax(significant_neurons)
    
    # Check if the neuron_idx is in the sequence order
    if neuron_idx not in seq_order:
        raise ValueError(f"Neuron {neuron_idx} is not in the sequence order.")
    
    # Copy the occurrence matrix for the selected neuron
    occ_template = occs[neuron_idx].copy()
    
    # Create a copy to store the probability template
    prob_template = occ_template.copy()
    
    # Normalize each row, keeping rows with all zeros unchanged
    for neuron_idx in range(prob_template.shape[0]):
        if neuron_idx not in seq_order:
            prob_template[neuron_idx, :] = 0  # Mask rows not in seq_order
        elif np.any(prob_template[neuron_idx, :] != 0):  # Skip rows with all zeros
            row_sum = prob_template[neuron_idx, :].sum()
            prob_template[neuron_idx, :] /= row_sum
    
    return prob_template


Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
Intel MKL WARNING: Support of Intel(R) Streaming SIMD Extensions 4.2 (Intel(R) SSE4.2) enabled only processors has been deprecated. Intel oneAPI Math Kernel Library 2025.0 will require Intel(R) Advanced Vector Extensions (Intel(R) AVX) instructions.
